# Chains, Prompt Templates and Output Parsers

In LangChain v1, `create_agent` is the default way to build an LLM app (notebooks 02-06). But when the steps are **fixed and known in advance** (no tool loop, no decisions), a **chain** is simpler, cheaper and easier to test.

| | Agent (`create_agent`) | Chain (`prompt \| model \| parser`) |
|---|---|---|
| Who decides the next step? | the model | you, in code |
| Tools | yes | not needed |
| Best for | open-ended tasks | fixed pipelines: extract, transform, explain |

**Use case:** a calculator that reads a **word problem**, extracts the calculation, computes it in Python, then explains the answer.


## Setup

In [1]:
%%capture
%pip install --upgrade langgraph langchain langchain-aws langchain-community
%pip install --upgrade python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Load AWS credentials from .env (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, AWS_DEFAULT_REGION)
# boto3 / langchain-aws pick them up from the environment automatically.
load_dotenv()


True

In [3]:
from langchain_aws import ChatBedrockConverse

model = ChatBedrockConverse(
    model="amazon.nova-lite-v1:0",
    temperature=0,
)


## Prompt templates

`ChatPromptTemplate` is a reusable prompt with `{placeholders}`. (Notebook 03 did the same thing at agent level with `@dynamic_prompt`.)

In [14]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a maths tutor. Explain in a {style} style in english."),
    ("human", "{question}"),
])

# A template is a Runnable: invoke it to see the messages it produces (no model call yet)
prompt.invoke({"style": "friendly", "question": "What is 12 plus 8?"}).to_messages()


[SystemMessage(content='You are a maths tutor. Explain in a friendly style in english.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 12 plus 8?', additional_kwargs={}, response_metadata={})]

## Your first chain with `|`

`prompt | model | StrOutputParser()` is **LCEL** (LangChain Expression Language). Each piece is a *Runnable*; the output of one is the input of the next.

`StrOutputParser` turns the model's `AIMessage` into a plain string.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# model json -> StrOutputParser-> text
explain_chain = prompt | model | StrOutputParser()

explain_chain.invoke({"style": "pirate", "question": "What is 12 plus 8?"})


"Ahoy, matey! Gather 'round and let ol' Captain Maths spin ye a tale o' numbers. Ye be askin' what be 12 plus 8, eh? Well, shiver me timbers, it's as simple as countin' yer gold doubloons!\n\nFirst, ye start with 12 pieces o' eight, arrr! Now, ye add 8 more to that treasure. So, ye count up from 12, like this: 13, 14, 15, 16, 17, 18, 19, and 20. \n\nSo, 12 plus 8 be equal to a full 20 pieces o' eight! Arrr, that's a fine haul, indeed!"

## Output parsers

Parsers turn model text into Python data.

* `JsonOutputParser`: text -> `dict`
* `PydanticOutputParser`: text -> validated Pydantic object. `get_format_instructions()` gives you the text to put in the prompt.

> In v1 agents use `response_format` instead (notebook 06). Parsers are for **chains**.

In [6]:
from langchain_core.output_parsers import JsonOutputParser

json_chain = (
    ChatPromptTemplate.from_template(
        "Return ONLY a JSON object with the keys 'expression' and 'answer' for: {question}"
    )
    | model
    | JsonOutputParser()
)

json_chain.invoke({"question": "What is 12 plus 8?"})


{'expression': '12 + 8', 'answer': 20}

In [7]:
from typing import Literal
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser


class Expression(BaseModel):
    operation: Literal["add", "subtract", "multiply", "divide"]
    a: float
    b: float


parser = PydanticOutputParser(pydantic_object=Expression)

extract_prompt = ChatPromptTemplate.from_messages([
    ("system", "Turn the word problem into ONE calculation.\n{format_instructions}"),
    ("human", "{problem}"),
]).partial(format_instructions=parser.get_format_instructions())

extract_chain = extract_prompt | model | parser

extract_chain.invoke({"problem": "Sam has 12 apples and buys 8 more. How many apples does Sam have?"})


Expression(operation='add', a=12.0, b=8.0)

## Runnable workflows

`RunnableLambda` wraps any Python function as a step, so plain code (here: the calculation) sits inside the chain.

In [8]:
from langchain_core.runnables import RunnableLambda

OPERATIONS = {
    "add": lambda a, b: a + b,
    "subtract": lambda a, b: a - b,
    "multiply": lambda a, b: a * b,
    "divide": lambda a, b: a / b,
}


def calculate(expr: Expression) -> float:
    if expr.operation == "divide" and expr.b == 0:
        raise ValueError("cannot divide by zero")
    return OPERATIONS[expr.operation](expr.a, expr.b)


calc_chain = extract_chain | RunnableLambda(calculate)

calc_chain.invoke({"problem": "A box holds 6 eggs. How many eggs are in 7 boxes?"})


42.0

### Keep the earlier values with `RunnablePassthrough.assign`

`assign` adds a new key to the data flowing through the chain **without losing the old keys**, so the last prompt can use the problem, the expression and the result together.

In [9]:
from langchain_core.runnables import RunnablePassthrough

answer_prompt = ChatPromptTemplate.from_template(
    "Problem: {problem}\n"
    "Calculation: {expression}\n"
    "Result: {result}\n\n"
    "Give the answer to the problem in one friendly sentence."
)

workflow = (
    RunnablePassthrough.assign(expression=extract_chain)
    | RunnablePassthrough.assign(result=lambda x: calculate(x["expression"]))
    | answer_prompt
    | model
    | StrOutputParser()
)

workflow.invoke({"problem": "A box holds 6 eggs. How many eggs are in 7 boxes?"})


'There are 42 eggs in 7 boxes.'

### Run steps side by side with `RunnableParallel`

In [10]:
from langchain_core.runnables import RunnableParallel

short_chain = ChatPromptTemplate.from_template("In one sentence: {problem}") | model | StrOutputParser()
steps_chain = ChatPromptTemplate.from_template("Solve step by step: {problem}") | model | StrOutputParser()

both = RunnableParallel(short=short_chain, steps=steps_chain)

result = both.invoke({"problem": "What is 15% of 80?"})
print("SHORT:", result["short"])
print("STEPS:", result["steps"])


SHORT: 15% of 80 is 12.
STEPS: To find 15% of 80, you can follow these steps:

1. Convert the percentage to a decimal. To do this, divide the percentage by 100.
   \[
   15\% = \frac{15}{100} = 0.15
   \]

2. Multiply the decimal by the number you want to find the percentage of.
   \[
   0.15 \times 80
   \]

3. Perform the multiplication.
   \[
   0.15 \times 80 = 12
   \]

So, 15% of 80 is 12.


## Invocation styles

Every Runnable has the same methods.

| Method | Use |
|---|---|
| `invoke` | one input, wait for the answer |
| `stream` | tokens as they arrive |
| `batch` | many inputs at once (runs in parallel) |
| `ainvoke` / `astream` / `abatch` | async versions for web servers |

In [11]:
problems = [
    {"problem": "What is 12 plus 8?"},
    {"problem": "A box holds 6 eggs. How many eggs are in 7 boxes?"},
    {"problem": "Split 90 euros between 4 friends."},
]

calc_chain.batch(problems, config={"max_concurrency": 3})


[20.0, 42.0, 22.5]

In [12]:
for chunk in explain_chain.stream({"style": "friendly", "question": "Why is dividing by zero impossible?"}):
    print(chunk, end="", flush=True)


Hey there! Let's dive into this interesting topic together. Imagine you have a bunch of cookies, and you want to share them with your friends. If you have 10 cookies and 2 friends, you can easily figure out that each friend gets 5 cookies because \(10 \div 2 = 5\).

Now, let's think about dividing by zero. Suppose you have 10 cookies, but instead of sharing them with friends, you want to share them with zero friends. How many cookies would each of those zero friends get? This is where things get tricky!

Here's a way to think about it: if you divide something by zero, you're asking how many times zero can fit into that something. But zero can't fit into anything because it's like trying to fit an infinite number of zeros into a finite number. It doesn't make sense!

Another way to see this is to think about the opposite of division, which is multiplication. If you have \(x \div 0 = y\), it means \(0 \times y = x\). But no matter what number you pick for \(y\), multiplying it by zero wi

In [13]:
# Notebooks run an event loop, so top-level await works. In a .py script use asyncio.run(...)
await calc_chain.ainvoke({"problem": "What is 12 plus 8?"})


20.0

## Summary

* `ChatPromptTemplate` -> reusable prompt with placeholders
* `|` -> chain Runnables: `prompt | model | parser`
* Parsers -> text to `str`, `dict` or a Pydantic object
* `RunnableLambda` / `RunnablePassthrough.assign` / `RunnableParallel` -> mix plain Python and models in one workflow
* `invoke`, `stream`, `batch`, `ainvoke` -> same methods on every Runnable
